# ICE Outcomes: Data Exploration & Cleaning

### Data Pre-Cleaning and Sampling
---

Before performing data cleaning and preprocessing in Python, we first used Google Cloud Platform and BigQuery to organize and pre-clean the raw immigration datasets.

The raw dataset was structured as separate tables for each year (2012–2023) and by category, including Arrests, Decisions, Detentions, and Removals. For each year, we combined these four tables into a single master table (e.g., master_50k_12, master_50k_2013, etc.) using BigQuery joins and queries. This allowed us to consolidate all relevant information for each case into one table per year.

After creating the yearly master tables, we performed sampling in BigQuery. We used a simple random sampling method to select 50,000 unqiue identifiers and their records per year. During sampling, we filtered for records where the target variable (`final_order`, Yes/No) was not NULL to ensure the data would be usable for later analysis and modeling.

After this preprocessing and sampling step in BigQuery, we exported the sampled data and continued further data cleaning, preprocessing, and analysis in Python.

### Data Loading & Flattening: 2012 - 2023 deportation data
--- 

In [2]:
# import modules
import pandas as pd
from google.cloud import bigquery

# connet to BigQuery
client = bigquery.Client(project="ice-data-project")

#### 2012 Deportation Data

In [3]:
# 50k SRS 2012 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_12`
"""

deportations_12 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_12[col] = deportations_12[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_12_flat = pd.json_normalize(deportations_12[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_12 = pd.concat([deportations_12, deportations_12_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_12 = deportations_12.drop(columns=[col])
print("\nData successfully flattened!")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [4]:
deportations_12.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,d0451cac01101d02e3b5c236e6b7432e5eff2148,2012-02-29,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),d0451cac01101d02e3b5c236e6b7432e5eff2148,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,722038e8783f0111938ba418485b72719130bb42,2012-06-12,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),722038e8783f0111938ba418485b72719130bb42,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,f196886fb97872592bce7642b1c1ea79722672b5,2012-08-23,CAP Local Incarceration,(b)(6)(b)(7)(c),None,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),f196886fb97872592bce7642b1c1ea79722672b5,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,764bb6cc37b909a941aeebe9ae5f813bfe450c87,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,c269a353ec849af9c41c95409e30fe9b960889c7,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


In [5]:
# List all columns
print("Columns in the dataframe:")
print(deportations_12.columns.tolist())

# Number of rows and columns
print(f"\nNumber of rows: {deportations_12.shape[0]}")
print(f"Number of columns: {deportations_12.shape[1]}")

# Number of nulls per column, sorted descending
null_counts = deportations_12.isnull().sum().sort_values(ascending=False)
print("\nTop columns by number of null values:")
print(null_counts[null_counts > 100000])  # only show columns with at least 1 null

Columns in the dataframe:
['Anonymized Identifier', 'arrests_Apprehension Date', 'arrests_Apprehension Method', 'arrests_Arrest Created By', 'arrests_Case ID', 'arrests_Subject ID', 'arrests_Alien File Number', 'arrests_Anonymized Identifier', 'decisions_RCA_AOR', 'decisions_RCA_DCO', 'decisions_A_NUMBER', 'decisions_SUBJ_ID', 'decisions_LAST_NAME', 'decisions_FIRST_NAME', 'decisions_ALERT_CODE', 'decisions_ACTIVE_INACTIVE', 'decisions_SUBMISSION_DATE', 'decisions_STATUS_CODE', 'decisions_RISK_TO_PUBLIC_SAFETY', 'decisions_RISK_OF_FLIGHT', 'decisions_SPECIAL_VULNERABILITY', 'decisions_RCA_CASE_NUMBER', 'decisions_CASE_CAT_AT_RCA_DECISION', 'decisions_FO_AT_RCA_DECISION', 'decisions_FO_DATE_AT_RCA_DECISION', 'decisions_REMOVAL_LIKELY_AT_RCA_DECISION', 'decisions_MAN_DET_PER_STAT_ALLEG', 'decisions_RCA_DECISION_TYPE', 'decisions_RCA_RECOMMENDATION', 'decisions_RCA_BOND_RECOMMENDATION', 'decisions_OFFICER_ID', 'decisions_OFFICER_AGREE_DISAGREE', 'decisions_SUPERVISOR_ID', 'decisions_SUPER

In [6]:
# Count unique values in the 'Anonymized Identifier' column
unique_ids = deportations_12['Anonymized Identifier'].nunique()
print(f"Number of unique Anonymized Identifiers: {unique_ids}")

# Count occurrences of each identifier
id_counts = deportations_12['Anonymized Identifier'].value_counts()
print(id_counts.head(10))  # show top 10 most frequent

Number of unique Anonymized Identifiers: 49999
Anonymized Identifier
0c3b60a99bb8cde4842ea33406925bd86415dc6e    324
4b24ab94116b4f9dc2526df4b8bb390f802f5c2d    240
f7b6851db74ee0632fdda3b48d7122839a694bbc    132
5e7b18264fd9f0b8b94565b960deb8f134395915    120
64ed1e275383b5bcaaaf19e75de28d4af30a9769    110
77cea3587cfcbc081e74162fc14579880d7d4161    108
a8a126df5f7304f860f817a5c97d1e81bc232a94    105
03345c5ab1d75f0f544610fd0bb84868058c2493    104
c4b538597d43914f82e1bc54c63e8dc936b392eb     90
24501b2bbbb8fa92e0c5920b123f9eb52918cc61     84
Name: count, dtype: int64


#### 2013 Deportation Data

In [7]:
# 50k SRS 2013 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_13`
"""

deportations_13 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_13[col] = deportations_13[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_13_flat = pd.json_normalize(deportations_13[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_13 = pd.concat([deportations_13, deportations_13_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_13 = deportations_13.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [8]:
deportations_13.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,6fd8b4caafbacbd1edf1551979c18f10b684a33b,2013-01-24,287(g) Program,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),6fd8b4caafbacbd1edf1551979c18f10b684a33b,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,1a70f48896990c255de0f1991c6321f5d1fc9117,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,842a00da4047c8f0cbb6532d61c52c6c2ca42e99,2012-10-03,CAP Local Incarceration,(b)(6)(b)(7)(c),None,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),842a00da4047c8f0cbb6532d61c52c6c2ca42e99,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,842a00da4047c8f0cbb6532d61c52c6c2ca42e99,2012-10-03,CAP Local Incarceration,(b)(6)(b)(7)(c),None,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),842a00da4047c8f0cbb6532d61c52c6c2ca42e99,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,842a00da4047c8f0cbb6532d61c52c6c2ca42e99,2013-01-22,CAP State Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),842a00da4047c8f0cbb6532d61c52c6c2ca42e99,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2014 Deportation Data

In [9]:
# 50k SRS 2014 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_14`
"""

deportations_14 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_14[col] = deportations_14[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_14_flat = pd.json_normalize(deportations_14[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_14 = pd.concat([deportations_14, deportations_14_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_14 = deportations_14.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [10]:
deportations_14.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,c973c05fa7745f87c4dbb713759f90535b17a335,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,670f91aa8596a98964f4069961e437a95919ed1b,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,8e62aaa75456081f665c6260fa750b76ed84553c,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,f86bc76347d7bc79769c0ae7bd3fc036b3c45ba8,2013-10-23,CAP Federal Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),f86bc76347d7bc79769c0ae7bd3fc036b3c45ba8,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,89f85f66658e11a56079da48068c6523bdd81d7d,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2015 Deportation Data

In [11]:
# 50k SRS 2015 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_15`
"""

deportations_15 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_15[col] = deportations_15[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_15_flat = pd.json_normalize(deportations_15[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_15 = pd.concat([deportations_15, deportations_15_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_15 = deportations_15.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [12]:
deportations_15.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,87f4668b9d2f22d0a91acb3cb3bfb660de3a8e9a,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,348cbbf6724480014158fa25250bb3cf0934b91f,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,a5e47977f2a07b40c1c5d3878b0b80c72149e94f,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,ab726f7d23d36b120220871bf476c1253cc59890,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,764e28376866f0cfc38e815522bf0f4f2e99f86e,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2016 Deportation Data

In [13]:
# 50k SRS 2016 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_16`
"""

deportations_16 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_16[col] = deportations_16[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_16_flat = pd.json_normalize(deportations_16[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_16 = pd.concat([deportations_16, deportations_16_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_16 = deportations_16.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [14]:
deportations_16.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,d2a55dcee700d2ae735951608a7f754148a57c68,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,c3d357816e649c11ca7e8f32ecb6c516a01cb23e,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,6ec2d10c149a95228a5b834552f87171ac62caf4,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,d318d527c7e021ad2adbee6fb3b3e6bdce767641,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,3af0e355532dd3d98c942c8e6611c39102dcf3e5,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2017 Deportation Data

In [15]:
# 50k SRS 2017 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_17`
"""

deportations_17 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_17[col] = deportations_17[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_17_flat = pd.json_normalize(deportations_17[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_17 = pd.concat([deportations_17, deportations_17_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_17 = deportations_17.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [16]:
deportations_17.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,48642ead9e3364cdce3f29a23a4f9399927e4fed,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,fc386240402229034c0fcb11c2ce89cfb82269dc,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,fa2ce674e31fb126d2f65e336cdd1f4956e45c81,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,434f3154aa599c833c1a3ac140f685326a671256,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,91a7fb2a3532a68463359f855acd24bd694dd14b,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2018 Deportation Data

In [17]:
# 50k SRS 2018 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_18`
"""

deportations_18 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_18[col] = deportations_18[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_18_flat = pd.json_normalize(deportations_18[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_18 = pd.concat([deportations_18, deportations_18_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_18 = deportations_18.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [18]:
deportations_18.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,cd7404e56c3b673baf35f613125532bd98f0d2ef,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,c0e23f0039afcb2b7d70f5e2f06fba236384075a,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,98ee1cd649d2a26610cba8e6ca507ca7d6ebcaaf,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,0e1fa26c765520978eefcc8187c442d3a92f5fb9,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,bdcdd8a1411c0e6f2a78446c6e17985e01db3836,2018-08-29,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),bdcdd8a1411c0e6f2a78446c6e17985e01db3836,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2019 Deportation Data

In [19]:
# 50k SRS 2019 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_19`
"""

deportations_19 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_19[col] = deportations_19[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_19_flat = pd.json_normalize(deportations_19[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_19 = pd.concat([deportations_19, deportations_19_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_19 = deportations_19.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [20]:
deportations_19.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,c86db3d9a63784bbce9baf0e15b09dc1f869a0d0,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,3a7a9baed000326b88abb68a5b9a2b774258e168,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,6630ea66bf15256d10a1e59f8ec3038b88155116,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,c1c86459219449f62ade316aad9d8879e59324d8,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,481318028ec7a89adb7f8dc2e8b7d6cc604dac3e,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2020 Deportation Data

In [21]:
# 50k SRS 2020 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_20`
"""

deportations_20 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_20[col] = deportations_20[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_20_flat = pd.json_normalize(deportations_20[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_20 = pd.concat([deportations_20, deportations_20_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_20 = deportations_20.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [22]:
deportations_20.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,94cc2236510af980234196d5787a25d44189ae59,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,542762f119553e49127bbeebbaf1f061910b065b,2020-09-01,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),542762f119553e49127bbeebbaf1f061910b065b,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,a30a74d9b69add3dd79e39420995435d4317c29a,2020-02-28,CAP Federal Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),a30a74d9b69add3dd79e39420995435d4317c29a,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,0663e91199b769ecad2277528a282de46b9161b5,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,67079a8dc1c79f4d98f90143fa483d6c218a3cb2,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2021 Deportation Data

In [23]:
# 50k SRS 2021 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_21`
"""

deportations_21 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_21[col] = deportations_21[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_21_flat = pd.json_normalize(deportations_21[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_21 = pd.concat([deportations_21, deportations_21_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_21 = deportations_21.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [24]:
deportations_21.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,d9a67a82d6f078a5591424caf84b7d67d21ef388,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,5e6ad00ac1ca41ba7a9ca72955383db7d79e4e60,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,1d95d2f193ef2338ebc168a78144e8fff0a555ae,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,2c89d9fe27bfad63e49262549321f7dc8bc78fb4,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,2238e4080589810793b4ce5240bc36530c27691f,2021-09-15,Non-Custodial Arrest,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),2238e4080589810793b4ce5240bc36530c27691f,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


#### 2022 Deportation Data

In [25]:
# 50k SRS 2022 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_22`
"""

deportations_22 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_22[col] = deportations_22[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_22_flat = pd.json_normalize(deportations_22[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_22 = pd.concat([deportations_22, deportations_22_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_22 = deportations_22.drop(columns=[col])
print("\nData successfully flattened!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [26]:
deportations_22.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,3448f9b5dc1082e198dc780b23c0c4d817aa0cb9,2022-07-19,Non-Custodial Arrest,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),3448f9b5dc1082e198dc780b23c0c4d817aa0cb9,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,fbc20aaa239d6922d3337d83e97c7cb73e8cec1b,2022-04-20,Non-Custodial Arrest,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),fbc20aaa239d6922d3337d83e97c7cb73e8cec1b,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,ca78c8fb1e233d7f37914a1bd74b3937a4b77394,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,3f287a86e3eef465d721d7fed88a443d332fefdd,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,d8f9d3f226db776779f5fd095ffba4ec8351a6bf,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


### Data Loading & Flattening: 2023 - 2025 deportation data
--- 

#### 2023 Deportation Data

#### 2024 Deportation Data

#### 2025 Deportation Data

### Comparison of Sample Representation vs Population Data
---

### Handling Schema Differences Across Data Releases
---

This project uses ICE enforcement data from two different public data releases that cover different time periods and were structured differently.

The most recent data release covers ICE enforcement actions from September 2023 through mid-October 2025 and includes tables for Arrests, Detainers, and Detentions. However, this release does not include Removals or Encounters tables due to potential data errors identified in those datasets. Earlier removals data remains available from the previous data release.

To enable long-term analysis, we also used ICE enforcement data from October 2011 through September 2023 which includes four major enforcement tables: Arrests, Detentions, Removals, and Decision History. These tables are linked using anonymized identifiers for each individual.

### Combining Datasets Across Years
---

Each fiscal year dataset was stored as a separate DataFrame. In order to perform longitudinal analysis and build predictive models, we combined all datasets from 2012–2025 into a single dataset.

Before combining, we added a `year` column to each dataset to indicate the fiscal year associated with each record. We then concatenated all datasets into one master DataFrame containing all years of ICE records.

In [ ]:
# adding year column to each dataset
deportations_13['year'] = 2013
deportations_14['year'] = 2014
deportations_15['year'] = 2015
deportations_16['year'] = 2016
deportations_17['year'] = 2017
deportations_18['year'] = 2018
deportations_19['year'] = 2019
deportations_20['year'] = 2020
deportations_21['year'] = 2021
deportations_22['year'] = 2022
# need to do this for 23, 24, 25 when we have the data for those years

In [ ]:
# combining all year into one dataset
all_deportations = pd.concat([
    deportations_12,
    deportations_13,
    deportations_14,
    deportations_15,
    deportations_16,
    deportations_17,
    deportations_18,
    deportations_19,
    deportations_20,
    deportations_21,
    deportations_22

    # need to do this for 23, 24, 25 when we have the data for those years
], ignore_index=True)

print(all_deportations.shape)

/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_75324/3126329612.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_deportations = pd.concat([
/var/folders/_s/x14mjmjn7qj5qh8bhxln436r0000gn/T/ipykernel_75324/3126329612.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_deportations = pd.concat([


(1991611, 111)


In [33]:
print(all_deportations.columns.tolist())

['Anonymized Identifier', 'arrests_Apprehension Date', 'arrests_Apprehension Method', 'arrests_Arrest Created By', 'arrests_Case ID', 'arrests_Subject ID', 'arrests_Alien File Number', 'arrests_Anonymized Identifier', 'decisions_RCA_AOR', 'decisions_RCA_DCO', 'decisions_A_NUMBER', 'decisions_SUBJ_ID', 'decisions_LAST_NAME', 'decisions_FIRST_NAME', 'decisions_ALERT_CODE', 'decisions_ACTIVE_INACTIVE', 'decisions_SUBMISSION_DATE', 'decisions_STATUS_CODE', 'decisions_RISK_TO_PUBLIC_SAFETY', 'decisions_RISK_OF_FLIGHT', 'decisions_SPECIAL_VULNERABILITY', 'decisions_RCA_CASE_NUMBER', 'decisions_CASE_CAT_AT_RCA_DECISION', 'decisions_FO_AT_RCA_DECISION', 'decisions_FO_DATE_AT_RCA_DECISION', 'decisions_REMOVAL_LIKELY_AT_RCA_DECISION', 'decisions_MAN_DET_PER_STAT_ALLEG', 'decisions_RCA_DECISION_TYPE', 'decisions_RCA_RECOMMENDATION', 'decisions_RCA_BOND_RECOMMENDATION', 'decisions_OFFICER_ID', 'decisions_OFFICER_AGREE_DISAGREE', 'decisions_SUPERVISOR_ID', 'decisions_SUPERVISOR_AGREE_DISAGREE', 'de

**Administrative Subsets**
To analyze differences in immigration outcomes across presidential administrations, 
we divided the dataset into subsets based on administration period. The administrations 
were defined based on fiscal years:

- Obama Administration: 2009-2012, 2013–2016 (have data starting 2012)
- Trump Administration: 2017–2020, 2025-2028
- Biden Administration: 2021–2024
- Trump Current Administration: 2025-2028

Creating these subsets allows us to compare trends, model outcomes separately, 
and analyze whether policy periods influenced final order outcomes.

In [ ]:
def get_administration(year):
    if year <= 2016:
        return 'Obama'
    elif year <= 2020:
        return 'Trump'
    elif year <= 2024:
        return 'Biden'
    else:
        return 'Trump' # years after 2024 will be assumed to be under Trump administration for now, but this can be updated later when we have more data

all_deportations['administration'] = all_deportations['year'].apply(get_administration)

### Feature Engineering
---

After combining the datasets, we created several new variables to support analysis and predictive modeling. These features include age, administration period, and time-based variables such as the number of days between arrest, detention, final order, and removal.

These engineered features help capture timelines and policy periods that may 
influence immigration case outcomes.

### Handling Missing Values
---
The dataset contained missing values across several variables due to incomplete records and differences in reporting across years. To prepare the data for modeling, we handled missing values by ____________. 

This approach allowed us to retain as many records as possible while ensuring the dataset remained usable for machine learning models.

### Target Variable Creation??? 
---
The primary outcome variable for this project is whether an individual received a final order of removal. Since final order information appeared in multiple tables, we combined the relevant columns into a single `final_order` variable and converted it into a binary variable (`final_order_binary`) for modeling purposes.

This binary variable will be used as the target variable in predictive models.

### Final Modeling Datasets
---

#### Master Dataset

#### Administration Subsets

### Exploratory Data Analysis
---

### Model
---

### Comparison of Obama vs Trump Models
---